[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/04_data_preprocessing/03_encoding_categorical.ipynb)

# Bài 03 — Mã hoá biến phân loại (categorical encoding)

**🎯 Mục tiêu**
- Phân biệt biến **định danh (nominal)** và **thứ bậc (ordinal)** và chọn cách mã hoá phù hợp.
- One-hot encoding (và các bẫy: dummy trap, category lạ), ordinal encoding.
- Biến **nhiều giá trị** (high cardinality): gộp category hiếm, frequency encoding, **target encoding** (và leakage của nó!), hashing.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/04_data_preprocessing

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

from ml_course.preprocessing import OneHotEncoderScratch

rng = np.random.default_rng(0)

## 1. Model chỉ hiểu số

Hầu hết model cần đầu vào là số. Biến phân loại phải được **mã hoá**. Cách mã hoá sai sẽ đưa vào model những quan hệ **không tồn tại**.

| Loại biến | Ví dụ | Có thứ tự? | Cách mã hoá |
|---|---|---|---|
| Nominal (định danh) | quận, phương thức thanh toán, màu | ❌ | one-hot; target/frequency encoding nếu nhiều giá trị |
| Ordinal (thứ bậc) | trình độ (THPT < ĐH < ThS), size (S < M < L) | ✅ | ordinal encoding **theo đúng thứ tự** |
| Binary | có/không | — | 0/1 |

## 2. Bẫy của Label Encoding với biến định danh

Gán `Quận 1 → 0, Quận 3 → 1, Quận 7 → 2, ...` ngụ ý "Quận 7 lớn hơn Quận 3" và "Quận 3 nằm giữa Quận 1 và Quận 7" — vô nghĩa với model tuyến tính / KNN. (Với model cây, ordinal encoding biến định danh đôi khi vẫn dùng được vì cây có thể tách nhiều lần, nhưng không tối ưu.)

## 3. One-hot encoding

Mỗi category thành một cột 0/1:

In [ ]:
df = pd.DataFrame({"payment": ["cash", "card", "e-wallet", "card", "cash"],
                   "size": ["M", "S", "L", "M", "XL"]})

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe.fit(df[["payment"]])
print(ohe.get_feature_names_out())
print(ohe.transform(df[["payment"]]))

**Category lạ khi dự đoán:** dữ liệu mới có thể chứa category chưa từng gặp khi train (vd: phương thức thanh toán mới "crypto"). `handle_unknown="ignore"` mã hoá nó thành vector toàn 0 thay vì báo lỗi làm sập hệ thống:

In [ ]:
print(ohe.transform(pd.DataFrame({"payment": ["crypto", "card"]})))

# Phiên bản from scratch của khoá học (src/ml_course/preprocessing.py)
scratch = OneHotEncoderScratch().fit(df["payment"])
print(scratch.categories_, scratch.transform(["crypto", "card"]))

**Dummy variable trap:** với $K$ category, $K$ cột one-hot luôn có tổng bằng 1 → **đa cộng tuyến hoàn hảo** với cột bias. Linear Regression **không regularization** giải bằng normal equation sẽ gặp ma trận suy biến (chương 02, bài 03). Cách xử lý: `drop="first"` (bỏ một cột làm mốc). Với model có regularization hay model cây, thường giữ đủ $K$ cột.

In [ ]:
ohe_drop = OneHotEncoder(drop="first", sparse_output=False).fit(df[["payment"]])
print(ohe_drop.get_feature_names_out(), "→ 'card' là mốc (toàn 0)")

⚠️ **`pd.get_dummies` không nhớ category của train** — áp dụng riêng cho train và test có thể tạo ra **số cột khác nhau** hoặc thứ tự khác nhau. Trong pipeline thật, dùng `OneHotEncoder` (fit trên train).

In [ ]:
train = pd.DataFrame({"payment": ["cash", "card", "e-wallet"]})
test = pd.DataFrame({"payment": ["card", "card"]})
print(pd.get_dummies(train).columns.tolist(), "vs", pd.get_dummies(test).columns.tolist(), "← lệch cột!")

## 4. Ordinal encoding — phải chỉ định thứ tự

Mặc định `OrdinalEncoder` sắp category theo **bảng chữ cái** → `L < M < S < XL` — sai! Luôn truyền `categories=` theo đúng thứ tự nghiệp vụ:

In [ ]:
wrong = OrdinalEncoder().fit_transform(df[["size"]]).ravel()
right = OrdinalEncoder(categories=[["S", "M", "L", "XL"]]).fit_transform(df[["size"]]).ravel()
print(pd.DataFrame({"size": df["size"], "mặc định (sai)": wrong, "đúng thứ tự": right}))

## 5. Biến nhiều giá trị (high cardinality)

Mã bưu điện, mã sản phẩm, tên thành phố... có thể có hàng nghìn giá trị → one-hot tạo hàng nghìn cột thưa, nhiều category chỉ xuất hiện vài lần (model không học được gì, dễ overfit).

### 5.1 Gộp category hiếm

In [ ]:
cities = pd.Series(rng.choice([f"city_{i}" for i in range(200)], 3000, p=np.r_[np.full(10, 0.07), np.full(190, 0.3 / 190)]))
counts = cities.value_counts()
rare = counts[counts < 20].index
grouped = cities.where(~cities.isin(rare), "OTHER")
print(f"{cities.nunique()} category → {grouped.nunique()} sau khi gộp các category < 20 lần thành 'OTHER'")

`OneHotEncoder(min_frequency=20, handle_unknown="infrequent_if_exist")` làm việc này tự động (và học ngưỡng từ train).

### 5.2 Frequency encoding

Thay category bằng **tần suất** xuất hiện của nó (học từ train). Đơn giản, 1 cột, hữu ích khi "phổ biến" có ý nghĩa.

In [ ]:
freq = cities.value_counts(normalize=True)
print(cities.map(freq).head())

### 5.3 Target encoding — mạnh nhưng nguy hiểm

Thay category bằng **trung bình của target** trong category đó (vd: tỉ lệ churn của thành phố). Rất mạnh cho biến nhiều giá trị, nhưng:

1. **Leakage:** nếu tính trung bình target trên chính các hàng đang train, mỗi hàng "nhìn thấy" nhãn của chính nó → model học thuộc. Category hiếm bị ảnh hưởng nặng nhất (category xuất hiện 1 lần → encoding = đúng nhãn của nó!).
2. **Category hiếm** cho trung bình rất nhiễu → cần **làm mượt (smoothing)** về trung bình chung:

$$\text{enc}(c) = \frac{n_c\cdot\bar y_c + m\cdot\bar y}{n_c + m}$$

**Cách làm đúng:** tính encoding cho tập train bằng **out-of-fold** (K-fold: mỗi fold được encode bằng thống kê từ các fold khác), còn test dùng thống kê từ toàn bộ train. `sklearn.preprocessing.TargetEncoder` làm đúng như vậy.

Thực nghiệm: một biến hoàn toàn **ngẫu nhiên** (không liên quan gì tới target) với 1000 giá trị:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder

n = 4000
noise_cat = rng.integers(0, 1000, n).astype(str)          # vô nghĩa
y = rng.integers(0, 2, n)                                   # target ngẫu nhiên → AUC thật = 0.5
tr, te = np.arange(3000), np.arange(3000, n)

# ❌ Naive: trung bình target tính trên chính dữ liệu train
means = pd.Series(y[tr]).groupby(noise_cat[tr]).mean()
naive_tr = pd.Series(noise_cat[tr]).map(means).to_numpy().reshape(-1, 1)
naive_te = pd.Series(noise_cat[te]).map(means).fillna(y[tr].mean()).to_numpy().reshape(-1, 1)
clf = LogisticRegression().fit(naive_tr, y[tr])
print(f"Naive target encoding  — AUC train = {roc_auc_score(y[tr], clf.predict_proba(naive_tr)[:, 1]):.3f}, "
      f"AUC test = {roc_auc_score(y[te], clf.predict_proba(naive_te)[:, 1]):.3f}")

# ✅ Out-of-fold + smoothing (sklearn TargetEncoder)
te_enc = TargetEncoder(random_state=0)
oof_tr = te_enc.fit_transform(noise_cat[tr].reshape(-1, 1), y[tr])   # fit_transform dùng cross-fitting
enc_te = te_enc.transform(noise_cat[te].reshape(-1, 1))
clf = LogisticRegression().fit(oof_tr, y[tr])
print(f"TargetEncoder (OOF)    — AUC train = {roc_auc_score(y[tr], clf.predict_proba(oof_tr)[:, 1]):.3f}, "
      f"AUC test = {roc_auc_score(y[te], clf.predict_proba(enc_te)[:, 1]):.3f}")

Nhìn phân phối giá trị encoding trên **train**, tách theo nhãn thật:

In [ ]:
import matplotlib.pyplot as plt

from ml_course.visualization import set_style

set_style()
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, enc, title in [(axes[0], naive_tr.ravel(), "Naive: trung bình target trên chính train"),
                       (axes[1], oof_tr.ravel(), "Out-of-fold (TargetEncoder)")]:
    for label, color in [(0, "C0"), (1, "C3")]:
        ax.hist(enc[y[tr] == label], bins=30, alpha=0.6, color=color, label=f"nhãn thật = {label}")
    ax.set_title(title)
    ax.set_xlabel("giá trị encoding")
axes[0].legend()
plt.suptitle("Biến vô nghĩa: naive encoding \"tách\" được hai lớp trên train — đó là nhãn bị rò rỉ vào feature")
plt.tight_layout()
plt.show()

Naive target encoding cho AUC train rất cao trên một biến **vô nghĩa** — model "học" được từ nhiễu do leakage, và điểm số train/validation sai lệch hoàn toàn. Out-of-fold encoding cho AUC ≈ 0.5 ở cả train lẫn test — đúng sự thật.

### 5.4 Feature hashing

Băm category vào một số cột cố định (`sklearn.feature_extraction.FeatureHasher`): không cần lưu từ điển, xử lý được category mới, đánh đổi bằng **va chạm** (hai category chung một cột). Phổ biến trong hệ thống quảng cáo/quy mô rất lớn.

## 6. Bảng tổng kết

| Tình huống | Gợi ý |
|---|---|
| Nominal, ít giá trị (< ~15) | One-hot (`handle_unknown="ignore"`) |
| Ordinal | OrdinalEncoder với `categories=` đúng thứ tự |
| Nominal, nhiều giá trị | gộp hiếm + one-hot, hoặc target encoding (OOF + smoothing), frequency encoding |
| Rất nhiều giá trị, streaming | hashing |
| Model cây gradient boosting hiện đại | CatBoost/LightGBM hỗ trợ categorical trực tiếp (chương 12) |
| Deep learning | embedding (chương 26, 32) |

## 🧠 Kiểm tra nhanh

1. Vì sao không nên mã hoá `district` thành 0, 1, 2, 3, 4 cho Linear Regression?
2. `handle_unknown="ignore"` giải quyết vấn đề gì khi triển khai?
3. Vì sao target encoding tính trên chính dữ liệu train gây leakage? Cách khắc phục?
4. `OrdinalEncoder()` mặc định mã hoá `["S", "M", "L"]` theo thứ tự nào?

## 📌 Tóm tắt

- Nominal → one-hot; ordinal → ordinal encoding đúng thứ tự.
- Dùng encoder của sklearn (fit trên train) thay vì `get_dummies` trên từng tập.
- High cardinality: gộp hiếm, frequency, target encoding **out-of-fold + smoothing**, hashing.

## 📚 Tài liệu tham khảo

- scikit-learn — Encoding categorical features: https://scikit-learn.org/stable/modules/preprocessing.html#encoding-categorical-features
- Micci-Barreca (2001) — *A preprocessing scheme for high-cardinality categorical attributes* (target encoding)